# Part 16 — RAG vs Long-Context vs CAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-16-rag-vs-long-context/rag_vs_long_context.ipynb)

*Part 1 asked why RAG exists. This part asks the harder follow-up: when do you even need retrieval? It closes the loop with a runnable cost model.*

📖 Read the essay: https://www.mefby.com/essays/rag-vs-long-context

🐍 Companion script: `rag_vs_long_context.py`

## What you'll build

For fifteen parts we assumed that to ground a model in your documents you must **retrieve**: embed the query, search an index, pull back a few chunks, paste them in. That was correct in 2023, when context windows held a few thousand tokens. In 2026 the frontier windows reach about **a million tokens** (enough to hold a few thousand pages at once), so a question that was unaskable three years ago is now live: if the whole corpus fits in the window, why retrieve from it at all?

The honest answer is *sometimes you should not*, but "just stuff everything in" is not free, because of the single most important fact in this part: **every token in the prompt is billed on every request**. This notebook is the **accountant** for that trade. You will build a small, honest cost model that prices three strategies for answering N queries against the same corpus:

- **RAG** → retrieve top-*k* and send only those chunks; cost tracks *k*, **not** corpus size;
- **long-context** → stuff the whole corpus in fresh on every request (the naive big-window baseline);
- **CAG** → stuff the corpus in **once** as a cacheable prefix, then reuse the cached state at a fraction of the price (Cache-Augmented Generation, [arXiv:2412.15605](https://arxiv.org/abs/2412.15605)).

Concretely, cell by cell, you'll build:

- the `Pricing` dataclass: the public *shape* of 2026 prompt caching (fresh = 1 unit, cache write 1.25x, cache read 0.1x);
- the `Workload` dataclass: one small, stable corpus answered many times;
- `cost_rag`, `cost_long_context`, `cost_cag`: three functions that price N queries, differing only in **which tokens are cacheable**;
- a 1-query table (CAG is the *worst*) and a 100-query table (CAG *wins*), the crossover finder, and a corpus-size sweep where the winner flips;
- two experiments that make the essay's pitfalls numeric: **kill the cache discount**, and **break the prefix** with volatile tokens;
- the **decision matrix** the whole part builds toward.

> **Runs with no network, no API key, and not even numpy.** This part is a pricing argument, not a model demo, so it is pure standard library. There is nothing to download and nothing to fall back to: every cell is deterministic arithmetic you can re-derive by hand. The numbers are **relative cost units**, not a vendor quote: plug in your provider's real per-token rates and the digits move, but the crossover *shape* does not.

## Setup

Two imports from the standard library do everything. `dataclass` gives us small, immutable, self-documenting bundles of numbers (`Pricing`, `Workload`); `replace` lets us make a tweaked copy of one for a sweep ("same workload but a bigger corpus") without mutating the original. `Optional` is just a type hint for the crossover finder, which may return `None`. No numpy, no model, no network.

In [ ]:
from dataclasses import dataclass, replace
from typing import Optional

print("standard library only (no numpy, no API key, no network)")

## Step 1 — The pricing, in relative cost units

Everything downstream rests on four numbers, and they encode the **public shape** of 2026 prompt caching on a frontier model:

- **fresh input = 1.0**: the unit. A token the model has never seen this call.
- **cache write = 1.25**: a small one-time premium the first time you lay down a stable, cacheable prefix.
- **cache read = 0.10**: every *subsequent* read of that prefix, an order of magnitude cheaper than fresh. **This 0.1x is the load-bearing wall**: it is the entire economic engine behind CAG. Without it, "preload once" would still cost full price on every query.
- **output = 5.0**: billed the same no matter how the input got there; carried for honesty so the totals are not misleadingly input-only.

These are *units*, not dollars. The point is the **ratio** (cached reads ~10x cheaper than fresh), not any one vendor's price sheet. `frozen=True` makes the dataclass immutable, so a stray edit can't silently corrupt a later calculation.

In [ ]:
@dataclass(frozen=True)
class Pricing:
    fresh_input: float = 1.0
    cache_write: float = 1.25   # one-time premium to write a stable prefix
    cache_read: float = 0.10    # every subsequent read of that prefix
    output: float = 5.0         # same for all strategies; carried for honesty


p = Pricing()
print(f"fresh={p.fresh_input}  write={p.cache_write}x  "
      f"read={p.cache_read}x  output={p.output}")
print(f"cache read is {p.fresh_input / p.cache_read:.0f}x cheaper than fresh "
      "(the whole CAG case in one ratio)")

## Step 2 — The workload: one small, stable corpus answered many times

The scenario CAG is built for: a *small, stable* knowledge base (a product's docs, a policy handbook, a single contract) answered over and over. The token counts are illustrative but in a realistic ratio: a few-thousand-token corpus, a short query, a top-*k* slice an order of magnitude smaller than the corpus, a couple-hundred-token answer, and a fixed instruction block.

One field deserves a flag now: **`volatile_tokens`** models a pitfall we'll exploit in Experiment 2. It is per-request content (a timestamp, a tool result) that slips *into* the cached prefix. Because it changes every call it can never cache, so it gets billed fresh every time, quietly defeating the point of caching. It defaults to `0` (a clean prefix); we leave it there until the experiment.

In [ ]:
@dataclass(frozen=True)
class Workload:
    corpus_tokens: int = 4000        # the whole knowledge base (small + stable)
    query_tokens: int = 40           # the user's question (volatile, never cached)
    retrieved_tokens: int = 600      # the top-k slice RAG actually sends
    answer_tokens: int = 250         # generated output, billed per strategy
    instruction_tokens: int = 200    # system prompt / fixed prefix
    volatile_tokens: int = 0         # per-request content that breaks the prefix


w = Workload()
print(f"corpus={w.corpus_tokens}  query={w.query_tokens}  "
      f"retrieved(top-k)={w.retrieved_tokens}  answer={w.answer_tokens}")
print(f"the top-k slice RAG sends is {w.corpus_tokens // w.retrieved_tokens}x "
      "smaller than the whole corpus (that gap is RAG's superpower)")

## Step 3 — Pricing RAG: a small slice, fresh every time

Now the heart of the model: three functions, one per strategy, each returning the cost to answer `n` queries against the **same** corpus. The whole subtlety is **which tokens are cacheable**. A stable prefix can be cached; anything that changes per request cannot.

RAG's defining property is that its per-query input is small and roughly constant: it tracks *k* (the chunks you retrieve), not the corpus size. But here is the catch the essay leans on: **RAG cannot cache its context.** Which chunks you retrieve changes from query to query, so there is no stable prefix to reuse: the retrieved slice and the query are *fresh* on every call. Only the fixed instruction block is a stable prefix (written once at 1.25x, then read at 0.1x thereafter). That is why RAG's cost barely moves as the corpus grows: it only ever sends *k* chunks, never the whole thing.

In [ ]:
def cost_rag(n: int, w: Workload, p: Pricing) -> dict:
    """Retrieve top-k per query; send only those chunks, fresh every time.
    Instructions are a stable prefix (cache once, read thereafter). The
    retrieved slice and the query are volatile, so they are fresh each call."""
    # instructions: one write, then (n-1) reads
    instr = w.instruction_tokens * p.cache_write + \
        w.instruction_tokens * p.cache_read * (n - 1)
    # retrieved slice + query: fresh on every one of the n calls
    per_query_fresh = (w.retrieved_tokens + w.query_tokens) * p.fresh_input
    input_cost = instr + per_query_fresh * n
    output_cost = w.answer_tokens * p.output * n
    return {"input": input_cost, "output": output_cost,
            "total": input_cost + output_cost}


print("RAG, 1 query: ", cost_rag(1, w, p))
print("RAG, 100 queries:", cost_rag(100, w, p))

## Step 4 — Pricing long-context: the whole corpus, fresh every time

The naive use of the big window: put the **entire corpus** into the prompt on every request and let the model read all of it. No index, no retrieval, no chunking, no embedding model: the simplest thing that could possibly work. The cost is the price of that simplicity: instructions **and** corpus are re-read at the *fresh* rate on every single call, because nothing is cached at all.

Note the consequence baked into the arithmetic: this cost is **linear in corpus size**, paid again on every query. A million-token prompt answered a thousand times is a billion input tokens, even though the corpus never changed. (Volatile tokens, if any, are fresh here too, but long-context never caches anyway, so they are just more input.)

In [ ]:
def cost_long_context(n: int, w: Workload, p: Pricing) -> dict:
    """Stuff the whole corpus in on every request at the FRESH rate (the naive
    'just use the big window' baseline -- no caching at all). Instructions and
    corpus are re-read fresh every single call."""
    per_query_fresh = (w.instruction_tokens + w.corpus_tokens + w.query_tokens +
                       w.volatile_tokens) * p.fresh_input
    input_cost = per_query_fresh * n
    output_cost = w.answer_tokens * p.output * n
    return {"input": input_cost, "output": output_cost,
            "total": input_cost + output_cost}


print("long_context, 1 query: ", cost_long_context(1, w, p))
print("long_context, 100 queries:", cost_long_context(100, w, p))
# 100 queries costs ~100x one query: nothing amortizes, every token is fresh.

## Step 5 — Pricing CAG: the whole corpus, cached once

The clever middle. The insight: if the corpus is the **same** on every request, you should not pay to process it from scratch every time. So you stuff the whole corpus in **once** as a stable, cacheable prefix (a *write* at 1.25x), and on every subsequent call you read it from cache at 0.1x. This is the prompt-caching shadow of Cache-Augmented Generation: the corpus is the stable prefix; only the query is fresh.

Read the arithmetic against long-context's. The corpus term changes from `corpus * fresh * n` (fresh, every call) to `corpus * write` once plus `corpus * read * (n-1)` thereafter. Same linear-in-corpus shape, but the slope is ~10x shallower because the recurring reads are at 0.1x. The `volatile_tokens` term is the trap: anything that changed per request sits *in* the prefix but is billed fresh every call (it can't cache), which is exactly the failure Experiment 2 will make numeric. With `volatile_tokens=0` (the clean case) this is the pure CAG win.

In [ ]:
def cost_cag(n: int, w: Workload, p: Pricing) -> dict:
    """Stuff the whole corpus in ONCE as a cacheable prefix (a write), then read
    it from cache on every subsequent call. The corpus is the stable prefix;
    only the query is fresh. Volatile tokens (if any) can't cache -> fresh."""
    prefix = w.instruction_tokens + w.corpus_tokens
    # the stable prefix: one write, then (n-1) cache reads
    prefix_cost = prefix * p.cache_write + prefix * p.cache_read * (n - 1)
    # the query is volatile -> fresh on every call
    query_cost = w.query_tokens * p.fresh_input * n
    # volatile tokens that slipped into the prefix: never cache -> fresh each call
    volatile_cost = w.volatile_tokens * p.fresh_input * n
    input_cost = prefix_cost + query_cost + volatile_cost
    output_cost = w.answer_tokens * p.output * n
    return {"input": input_cost, "output": output_cost,
            "total": input_cost + output_cost}


print("CAG, 1 query: ", cost_cag(1, w, p))
print("CAG, 100 queries:", cost_cag(100, w, p))
# At 1 query CAG paid the write premium for one read -> worst. By 100 it amortized.

## Step 6 — A registry and a comparison table

With the three pricing functions defined, register them under their names and write one small helper that prints them side by side for a given query count `n`, with each total expressed as a multiple of RAG's (the `vs RAG` column). Reading three numbers next to each other is what turns the arithmetic into a *decision*.

In [ ]:
STRATEGIES = {
    "rag": cost_rag,
    "long_context": cost_long_context,
    "cag": cost_cag,
}


def table(n: int, w: Workload, p: Pricing) -> None:
    rows = {name: fn(n, w, p) for name, fn in STRATEGIES.items()}
    base = rows["rag"]["total"]
    header = (f"  {'strategy':<14}  {'input cost':>10}  {'output cost':>11}  "
              f"{'total':>8}   {'vs RAG':>6}")
    print(header)
    print("  " + "-" * (len(header) - 2))
    for name, c in rows.items():
        ratio = c["total"] / base if base else float("inf")
        print(f"  {name:<14}  {c['input']:>10.1f}  {c['output']:>11.1f}  "
              f"{c['total']:>8.1f}   {ratio:>5.2f}x")


print("registry ready:", list(STRATEGIES))

## Step 7 — Cost to answer 1 query: CAG is the worst

Price all three at a single query. CAG is the **most expensive** option here, and that is not a bug, it is the point. You paid the cache-write premium and got exactly **one** read out of it, so you are carrying the cost of caching with none of the benefit. Judging CAG on one query is the wrong test; its entire value is amortization across many reuses of the same stable corpus.

In [ ]:
print("COST TO ANSWER 1 QUERY")
table(1, w, p)
print()
print("CAG is WORST at n=1: you paid the write premium and got one read.")

## Step 8 — Cost to answer 100 queries: CAG wins

Now the same workload at a hundred queries against that fixed corpus. The write premium has amortized across a hundred cheap 0.1x reads, and CAG drops **below** both naive long-context (which never caches and stays ~2.9x the bill) and even RAG. The corpus is small and stable, so CAG buys you long-context's simplicity *below* RAG's cost. Same code, same corpus: only the reuse count changed, and the verdict flipped.

In [ ]:
print("COST TO ANSWER 100 QUERIES (same stable corpus)")
table(100, w, p)
print()
print("CAG amortized the write across 100 reads and overtook RAG;")
print("long_context never caches, so it stays ~2.9x the bill.")

## Step 9 — The crossover: where exactly does CAG take the lead?

If CAG loses at 1 query and wins at 100, there is a crossover in between. `first_crossover` just walks `n` upward until the `beats` strategy's total drops below the `against` one: the smallest query count at which the verdict flips. Run it twice: CAG overtakes naive long-context almost immediately (it only has to beat a strategy that caches nothing), and overtakes RAG a couple dozen queries later (it has to amortize the write *and* out-pace RAG's tiny top-*k* slice). Those two numbers are the *boundary of the decision matrix*, computed rather than asserted.

In [ ]:
def first_crossover(beats: str, against: str, w: Workload, p: Pricing,
                    cap: int = 100000) -> Optional[int]:
    """Smallest n at which strategy `beats` has a lower total than `against`."""
    for n in range(1, cap + 1):
        if STRATEGIES[beats](n, w, p)["total"] < \
                STRATEGIES[against](n, w, p)["total"]:
            return n
    return None


print("CAG beats naive long-context after", first_crossover("cag", "long_context", w, p), "query")
print("CAG beats RAG after", first_crossover("cag", "rag", w, p), "queries")

## Step 10 — What moves the answer: the corpus-size sweep

The crossover above was for one corpus size. Now hold the traffic fixed at 100 queries and **sweep the corpus** from tiny to huge, printing the winner at each size. This is where the decision matrix comes from. `replace(w, corpus_tokens=corpus)` makes a copy of the workload with only the corpus changed.

Watch the flip: for a small corpus **CAG wins**, but as the corpus grows the winner becomes **RAG**. Three shapes are visible in the columns: long-context grows *linearly* with corpus size (it re-reads everything fresh every query, the brutal "just stuff it" tax); CAG grows the same way but ~10x slower (the cached reads are still 0.1x of an ever-larger corpus); and RAG stays nearly flat, because it only ever sends the top-*k* slice. That last fact is exactly why a **massive** corpus points at RAG.

In [ ]:
print("corpus-size sweep (100 queries, cache read = 0.10x)")
header = (f"  {'corpus tokens':>13}   {'rag total':>9}     "
          f"{'long_context':>12}     {'cag total':>9}     winner")
print(header)
print("  " + "-" * (len(header) - 2))
for corpus in (1000, 4000, 16000, 64000, 256000):
    ws = replace(w, corpus_tokens=corpus)
    totals = {name: fn(100, ws, p)["total"] for name, fn in STRATEGIES.items()}
    winner = min(totals, key=totals.get)
    print(f"  {corpus:>13}   {totals['rag']:>9.1f}     "
          f"{totals['long_context']:>12.1f}     "
          f"{totals['cag']:>9.1f}     {winner}")

## Step 11 — Experiment 1: kill the cache discount

The whole case for CAG rests on the cached read being a *fraction* of the fresh rate. So break that assumption and watch CAG collapse. `replace(p, cache_read=1.0)` makes a pricing where caching saves nothing, a read costs as much as a fresh token. Re-price 100 queries under both regimes.

Without the discount, CAG's total jumps up to roughly the naive long-context total: preloading once buys you nothing if every read is full price. This is the counterfactual world *before* cheap cached reads existed, and it is exactly why CAG was not a viable idea then. **The 0.1x discount is not a detail; it is the load-bearing wall.**

In [ ]:
p_nodisc = replace(p, cache_read=1.0)   # caching now saves nothing
print(f"  {'strategy':<14}     {'with cache (0.10x)':>18}     "
      f"{'no discount (1.00x)':>19}")
print("  " + "-" * 60)
for name, fn in STRATEGIES.items():
    with_cache = fn(100, w, p)["total"]
    no_disc = fn(100, w, p_nodisc)["total"]
    print(f"  {name:<14}     {with_cache:>18.1f}     {no_disc:>19.1f}")
print()
print(f"CAG with no discount ({cost_cag(100, w, p_nodisc)['total']:.1f}) "
      f"collapses toward long_context ({cost_long_context(100, w, p)['total']:.1f}).")

## Step 12 — Experiment 2: break the prefix with volatile tokens

Prompt caching is a **prefix match**: the provider can only reuse the cache if the bytes up to the cache point are byte-for-byte identical to a previous request. The most common way to wreck this is to let something *volatile* sneak into the prefix: a timestamp, a per-request id, a session token, or a changing tool result. Any of those changes the prefix bytes and invalidates the cache from that point on, so you quietly pay fresh-input rates for the corpus you thought was cached.

Make it numeric. Raise `volatile_tokens` (per-request content billed fresh every call, because it never caches) and watch CAG's 100-query advantage over RAG erode and then vanish. The lesson the column spells out: keep volatile content **last**, after the final cache point, never in the cached prefix.

In [ ]:
print(f"  {'volatile tokens':>15}   {'cag total':>9}     {'vs RAG':>6}     "
      "CAG still wins?")
print("  " + "-" * 58)
rag_total = cost_rag(100, w, p)["total"]
for vol in (0, 100, 300, 1000):
    wv = replace(w, volatile_tokens=vol)
    cag_total = cost_cag(100, wv, p)["total"]
    ratio = cag_total / rag_total
    wins = "yes" if cag_total < rag_total else "no"
    print(f"  {vol:>15}   {cag_total:>9.1f}     {ratio:>5.2f}x     {wins}")

## Step 13 — The decision matrix

Now we can answer the question the part opened with, and it comes down to two properties of your corpus plus cost as the tiebreaker: **how big** it is and **how often it changes**. Start with the disqualifiers, because they are absolute. If the corpus is **massive** (bigger than the window, or large enough that even cached reads of it are expensive), **fast-moving** (changing often enough that you'd recompute the cache constantly), or **private** (needs per-user access filtering), then you need **RAG**: the only strategy whose cost doesn't grow with the corpus, the only one that handles a corpus too big for any window, and the only one that can filter what each user sees before it reaches the prompt.

If the corpus clears those, it's **small and stable**, and the choice is between CAG and plain long-context. Lean **CAG** when it's small-but-not-tiny and you answer many queries (the write amortizes); lean plain **long-context** when it's so tiny the caching machinery isn't worth it, or queries are so rare you'd never amortize. A **mid-size** stable corpus that fits the window but is too big to want cached on every call goes to ordinary long-context. The table below is that rule, printed.

In [ ]:
print("THE DECISION MATRIX (size x volatility, cost as tiebreaker)")
print(f"  {'corpus shape':<36} ->  {'pick':<11} because")
print("  " + "-" * 76)
rows = [
    ("massive (bigger than the window)", "RAG",
     "only RAG's cost tracks k,", "not corpus size"),
    ("fast-moving (changes often)", "RAG",
     "a moving corpus recomputes", "the cache constantly"),
    ("private (per-user access filtering)", "RAG",
     "only retrieval can filter", "before the prompt"),
    ("small + stable + reused a lot", "CAG",
     "the write amortizes over", "cheap 0.1x reads"),
    ("tiny, or queried rarely", "long_context",
     "caching machinery is not", "worth the bother"),
    ("mid-size + stable", "long_context",
     "fits the window, too big", "to want cached per call"),
]
for shape, pick, why1, why2 in rows:
    print(f"  {shape:<36} ->  {pick:<11} {why1}")
    print(f"  {'':<36}     {'':<11} {why2}")

## What you learned

- The 2026 reality is that context windows reach about **a million tokens**, so for a small enough corpus you can skip retrieval. The catch is that **every prompt token is billed on every request**, so "just stuff it" trades a fits-in-the-window win for a paid-on-every-query cost.
- There are **three options**: **RAG** (send only the top-*k* slice; cost tracks *k*, not corpus size), **long-context** (send the whole corpus fresh every query; linear in corpus size forever), and **CAG** (preload once, cache the KV state, reuse it; long-context made affordable by caching).
- **Prompt-caching economics** are the bridge: a cached input token costs roughly **0.1x** the fresh rate, against a one-time **~1.25x** write premium. Structure the prompt as a stable cacheable prefix (instructions, then corpus) with the dynamic query last, and keep volatile tool results **out** of the prefix, or they quietly restore full pricing (Experiment 2).
- The **crossover** is computed, not asserted: CAG is the *worst* option at one query and the *best* by a hundred, and a corpus-size sweep flips the winner from **CAG** (small) to **RAG** (massive). That flip *is* the decision matrix.
- The closing lesson of the series, applied to the corpus rather than the query (Part 15): stop asking "RAG or not?" globally. Ask what **this** corpus wants (from its size, its volatility, and your traffic), and spend retrieval complexity only where it earns its keep.

### The Frontier Track, and where the series lands

This closes the loop on Part 1, which asked *why* retrieval, by asking *whether* you need it at all in 2026. The honest answer is: not always. A small, stable, reused corpus can be cached (CAG) or stuffed (long-context), while a massive, fast-moving, or private one still needs the full retrieval pipeline this series built. The production capstone still lives in **[Part 12](https://www.mefby.com/essays/rag-in-production)**; the best next step is the obvious one: point the cost model at *your* corpus and traffic, swap in your provider's real rates, and let the numbers tell you which strategy it wants.